# Sprint 3 - Tool Reasoning, Tool Errors, and MCP

This notebook follows LS9-LS12. You will design a direct tool layer, inspect schemas, handle malformed calls and runtime failures, run a multi-step tool loop, then connect to the helper MCP server and validate one MCP tool response.


## 1. Install the helper core from GitHub

Run this first in Colab. It uses `%pip` to install the shared helper core directly from the GitHub `main` branch before any helper imports.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --upgrade "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"

print("Installed helper core from GitHub main.")


## 2. Add your OpenRouter key and imports

The manual tool cells do not spend model credits. The agent cell does call OpenRouter, so store `OPENROUTER_API_KEY` in Colab Secrets when possible or use the hidden prompt.


In [ ]:
import os
from getpass import getpass


def load_openrouter_key() -> str:
    key = os.getenv("OPENROUTER_API_KEY")
    if key:
        return key

    try:
        from google.colab import userdata

        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None

    if not key:
        key = getpass("OpenRouter API key: ")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is required for this notebook's model calls.")

    os.environ["OPENROUTER_API_KEY"] = key
    return key


_ = load_openrouter_key()
print("OpenRouter API key loaded.")


In [ ]:
import json
import sys

from agent import ToolCallingAgent
from mcp_client import inspect_and_call_stdio_tool
from mcp_server import build_mcp_server, course_core_health, keyword_search_documents
from models import ChatModel
from openrouter import OpenRouterClient
from tools import ToolRegistry

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-3")


## 3. Register application tools

LS9 frames a tool as application code plus a schema. The model may propose a call, but the application validates and executes it.


In [ ]:
LESSON_NOTES = [
    "Structured output returns validated JSON for routing, UI state, and tool inputs.",
    "Hybrid retrieval combines ChromaDB semantic search with BM25 keyword matching.",
    "Reranking promotes the most useful retrieved passages before generation.",
    "HyDE rewrites a user query into a hypothetical answer document for retrieval.",
    "MCP exposes tools and resources through a common protocol for model clients.",
]


def lesson_lookup(query: str, top_k: int = 2):
    return keyword_search_documents(query=query, documents=LESSON_NOTES, top_k=top_k)


def lab_status(mode: str = "ok"):
    if mode == "timeout":
        raise TimeoutError("The lesson status service did not respond in time.")
    if mode == "missing":
        return {"ok": False, "message": "No lab status is available for that sprint."}
    return {"ok": True, "message": "The lab environment is ready."}


registry = ToolRegistry()
registry.register(
    name="lesson_lookup",
    description="Search short Module A lesson notes for relevant concepts.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query."},
            "top_k": {"type": "integer", "minimum": 1, "maximum": 5},
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    handler=lesson_lookup,
)
registry.register(
    name="lab_status",
    description="Check whether a sprint lab environment is ready.",
    parameters={
        "type": "object",
        "properties": {
            "mode": {"type": "string", "enum": ["ok", "missing", "timeout"]},
        },
        "required": [],
        "additionalProperties": False,
    },
    handler=lab_status,
)


## 4. Inspect the schemas sent to the model

A schema is the contract that makes a tool safe to call. Read the schema before trusting any proposed arguments.


In [ ]:
print(json.dumps(registry.to_openrouter_tools(), indent=2))


## 5. Execute a valid direct tool call

This is direct tool evidence for LS12: show the path from input, to validated arguments, to application result.


In [ ]:
good_call = {
    "id": "call_lookup_1",
    "type": "function",
    "function": {
        "name": "lesson_lookup",
        "arguments": json.dumps({"query": "hybrid search reranking", "top_k": 2}),
    },
}

result = registry.execute_tool_call(good_call)
print("ok:", result.ok)
print(result.content)
print(result.as_message())


## 6. Handle malformed and failed tool calls

LS10 asks you to classify failures from evidence. Different failure families need different recovery moves.


In [ ]:
problem_calls = [
    {
        "id": "call_bad_json",
        "type": "function",
        "function": {"name": "lesson_lookup", "arguments": "{\"query\": "},
    },
    {
        "id": "call_extra_arg",
        "type": "function",
        "function": {
            "name": "lesson_lookup",
            "arguments": json.dumps({"query": "MCP", "debug": True}),
        },
    },
    {
        "id": "call_timeout",
        "type": "function",
        "function": {"name": "lab_status", "arguments": json.dumps({"mode": "timeout"})},
    },
]


def recovery_hint(outcome):
    if outcome.retryable:
        return "Retry only if the operation is safe to repeat."
    if outcome.error_type == "ValueError":
        return "Repair the arguments or ask for clarification."
    return "Use a fallback or explain the missing capability."


failure_evidence = []
for call in problem_calls:
    outcome = registry.execute_tool_call(call)
    failure_evidence.append((call["id"], outcome.error_type, outcome.retryable, recovery_hint(outcome)))
    print(call["id"], "ok=", outcome.ok, "retryable=", outcome.retryable, "error=", outcome.error_type)
    print(outcome.content)
    print("recovery:", recovery_hint(outcome), "\n")


## 7. Let the model run a multi-step tool loop

The agent sends schemas, receives proposed calls, executes tools, appends tool messages, and asks the model for a final response.


In [ ]:
agent = ToolCallingAgent(
    client=client,
    tools=registry,
    model=ChatModel.GEMINI_31_FLASH_LITE,
    max_steps=4,
    system_prompt=(
        "You help students with Module A. Use lesson_lookup before answering questions "
        "about specific course concepts. Explain tool errors plainly if they happen."
    ),
)

run = agent.run("Which lesson note should I read to understand why reranking helps RAG?")
print(run.final_content)
print("\nTool calls executed:", len(run.tool_results))
for tool_result in run.tool_results:
    print(tool_result.name, tool_result.ok, tool_result.content[:200])


## 8. Connect to the MCP server

LS11 moves the same discipline to MCP: connect, inspect advertised capabilities, then call one safe tool. This cell starts the repo's MCP server over stdio.


In [ ]:
local_server = build_mcp_server("module-a-demo")
print("Built server object:", type(local_server).__name__)
print("Local health payload:", course_core_health())

mcp_demo = await inspect_and_call_stdio_tool(
    command=sys.executable,
    args=["-m", "mcp_server"],
    tool_name="keyword_search",
    arguments={
        "query": "model clients tools protocol",
        "documents": LESSON_NOTES,
        "top_k": 2,
    },
)

print("Connected server:", mcp_demo.server_name)
print("Advertised tools:", [tool.name for tool in mcp_demo.tools])
for tool in mcp_demo.tools:
    print(f"- {tool.name}: {tool.description}")


## 9. Validate the MCP response

An MCP connection proves only that the boundary exists. The host still validates the advertised tool and the returned content before using it downstream.


In [ ]:
tool_names = [tool.name for tool in mcp_demo.tools]
assert mcp_demo.server_name == "ms-ai-ml-helper-core"
assert "health" in tool_names
assert "keyword_search" in tool_names
assert mcp_demo.call.tool_name == "keyword_search"
assert mcp_demo.call.is_error is False
assert mcp_demo.call.content_texts

print("MCP call content:")
print("\n".join(mcp_demo.call.content_texts))


## 10. LS12 checkpoint evidence

A Sprint 3 defense should show one direct tool, one MCP capability, and one edge case where the app did not pretend failure was success.


In [ ]:
checkpoint_evidence = {
    "direct_tool": {
        "claim": "The app can run a local lesson_lookup tool.",
        "evidence": result.content,
    },
    "tool_failure": {
        "claim": "The app can classify malformed arguments and timeouts.",
        "evidence": failure_evidence,
    },
    "mcp_capability": {
        "claim": "The MCP server advertised keyword_search and returned usable content.",
        "evidence": {
            "server": mcp_demo.server_name,
            "tools": tool_names,
            "content": mcp_demo.call.content_texts,
        },
    },
}

print(json.dumps(checkpoint_evidence, indent=2, default=str))


## Optional local inspector

The notebook already connects to the MCP server over stdio. For the interactive MCP Inspector, run this locally:

```bash
git clone https://github.com/richhiey/ai-app-dev_Mod-A.git
cd ai-app-dev_Mod-A
python -m pip install -e ".[dev]"
mcp dev src/mcp_server.py
```
